In [1]:
import stim
import re
import os
from pyzx import *
import matplotlib.pyplot as plt
import networkx as nx
from pyvis.network import Network

def stim_qasm_comply(qasm: str) -> str:
    q = qasm
    q = re.sub(r'def\s+rx\(qubit q0\)\s*\{[^}]*\}\n+', '', q)
    q = re.sub(r'rx\s*\(\s*q\[(\d+)\]\s*\)\s*;', r'h q[\1];', q)
    q = re.sub(r'reset\s+q\[(\d+)\];', '', q)
    return q



In [2]:
tableau_5_code = stim.Tableau.from_stabilizers([
    stim.PauliString("XZZXI"),
    stim.PauliString("IXZZX"), 
    stim.PauliString("XIXZZ"),
    stim.PauliString("ZXIXZ")
], allow_underconstrained=True)

tableau_bit_code = stim.Tableau.from_stabilizers([
    stim.PauliString("ZZ_"),
    stim.PauliString("_ZZ"),
], allow_underconstrained=True)

# print(tableau_bit_code)
print(tableau_5_code)

k = 1
n = 5

stabilizers = []

for i in range (n - k):
    stabilizers.append(stim.PauliString(f"Z{i}") * stim.PauliString(n+k))

for i in range(n-k, n): 
    stabilizers.append(stim.PauliString(f"Z{i}*Z{i+k}") * stim.PauliString(n+k)) 
    stabilizers.append(stim.PauliString(f"X{i}*X{i+k}") * stim.PauliString(n+k)) 

# stabilizers.append(stim.PauliString("I" * (out - input))  + stim.PauliString("X" * 2*input))

print("Stabilizers:")
for s in stabilizers:
    print(s)

# tableau = stim.Tableau.from_stabilizers(stabilizers)

state = stim.TableauSimulator()
state.set_state_from_stabilizers(stabilizers)
state.do_tableau(tableau_5_code, range(n))

print(state.current_inverse_tableau().inverse())
t = state.current_inverse_tableau().inverse()

qasm = t.to_circuit(method="graph_state").to_qasm(open_qasm_version=3)

# print(tableau_5_code)
pyzx_circ = Circuit.from_qasm(stim_qasm_comply(qasm))
# draw(pyzx_circ  )
g = pyzx_circ.to_graph()
input_state = "0"*(n+k)
# draw(g, labels=True)
inputs = g.outputs()[n:n+k]
outputs = g.outputs()[:n]
g.apply_state(input_state)
# clifford_simp(g)
# g.normalize()
# g.auto_detect_io()
# draw(g, labels=True )

# draw(g, labels=True)
# c = Circuit.from_graph(g)

# draw(c)


t0 = tensorfy(g)
# g = GraphState(g)
t1 = tensorfy(g)
# g.to_canonical_form()
t2 = tensorfy(g)

compare_tensors(t0, t2)

d = to_universal_graph_representation(g, inputs)

d

# print(t0)

+-xz-xz-xz-xz-xz-
| -+ ++ -+ ++ --
| _X __ ZX _Z Z_
| _Z _X __ ZX _Z
| XZ _Z XX __ XX
| _X _Z _Z _X XZ
| Z_ ZX ZZ ZZ __
Stabilizers:
+Z_____
+_Z____
+__Z___
+___Z__
+____ZZ
+____XX
+-xz-xz-xz-xz-xz-xz-
| -+ ++ -+ ++ -- +-
| _X __ ZX _Z Z_ _Z
| _Z _X __ ZX _Z __
| XZ _Z XX __ XX _X
| _X _Z _Z _X XZ _X
| Z_ ZX ZZ ZZ __ __
| __ __ __ __ _Z ZX
(49,)


{'inputs': [0],
 'adjacency_list': [[2, 4, 5, 1, 3],
  [2, 0, 5],
  [1, 3, 0],
  [2, 4, 0],
  [5, 3, 0],
  [4, 0, 1]]}

In [3]:
d

{'inputs': [0],
 'adjacency_list': [[2, 4, 5, 1, 3],
  [2, 0, 5],
  [1, 3, 0],
  [2, 4, 0],
  [5, 3, 0],
  [4, 0, 1]]}

In [4]:

qasm_5qubit = tableau_5_code.to_circuit(method="elimination").to_qasm(open_qasm_version=3)

qasm_bit = tableau_bit_code = stim.Tableau.from_stabilizers([
    stim.PauliString("ZZ_"),
    stim.PauliString("_ZZ"),
], allow_underconstrained=True).to_circuit(method="elimination").to_qasm(open_qasm_version=3)

pyzx_circ = Circuit.from_qasm(qasm_5qubit)
g = pyzx_circ.to_graph()
input_state = "0"*(3) + "/"*2
g.apply_state(input_state)


g.auto_detect_io()

g = GraphState(g)
s1 = tensorfy(g)
# draw(g, labels=True )
# draw(g, labels=True)


g.to_canonical_form()
# g = g.state_to_map()
s2 = tensorfy(g)
# print(t2)
compare_tensors(t2, s2, preserve_scalar=False)

False

In [5]:
# qasm = tableau_5_code.to_circuit(method="elimination").to_qasm(open_qasm_version=3)
pyzx_circ = Circuit.from_qasm(stim_qasm_comply(qasm))
g = pyzx_circ.to_graph()
input_state = "0"*(6)
g.apply_state(input_state) 
g.auto_detect_io()
inputs = g.outputs()[5:6]
print(inputs)
to_universal_representation(g, inputs)


(49,)
(49,)


In [6]:

qasm_5qubit = stim.Tableau.from_stabilizers([
    stim.PauliString("XZZXI"),
    stim.PauliString("IXZZX"), 
    stim.PauliString("XIXZZ"),
    stim.PauliString("ZXIXZ")
], allow_underconstrained=True).to_circuit(method="elimination").to_qasm(open_qasm_version=3)

shor_code = stim.Tableau.from_stabilizers([
    stim.PauliString("ZZIIIIIII"),
    stim.PauliString("ZIZIIIIII"),
    stim.PauliString("IIIZZIIII"),
    stim.PauliString("IIIZIZIII"),
    stim.PauliString("IIIIIIZZI"),
    stim.PauliString("IIIIIIZIZ"),
    stim.PauliString("XXXXXXIII"),
    stim.PauliString("IIIXXXXXX"),
], allow_underconstrained=True).to_circuit().to_qasm(open_qasm_version=3)

qasm_7qubit = stim.Tableau.from_stabilizers([
    stim.PauliString("IIIXXXX"),
    stim.PauliString("IXXIIXX"),
    stim.PauliString("XIXIXIX"),
    stim.PauliString("IIIZZZZ"),
    stim.PauliString("IZZIIZZ"),
    stim.PauliString("ZIZIZIZ")
], allow_underconstrained=True).to_circuit().to_qasm(open_qasm_version=3) 


In [7]:

pyzx_circ = Circuit.from_qasm(qasm_5qubit)
g = pyzx_circ.to_graph()
input_state = "0"*(4) + "/"*1
g.apply_state(input_state)
d = to_universal_graph_representation(g)

TypeError: to_universal_graph_representation() missing 1 required positional argument: 'inputs'

In [ ]:
print(d["inputs"])

nxg = nx.Graph()
bounds = []
for v in range(len(d["adjacency_list"])):
    # v_type = v["t"]
    if v in d["inputs"]:
        color = "green"
    else:
        color = "blue"
    # if v_type != VertexType.BOUNDARY:
    nxg.add_node(v, 
                color=color, 
                )
    # else:
    # bounds.append(v["id"])

adj = d["adjacency_list"]

for i in range(len(adj)):
    for j in adj[i]:
        if i < j:
            nxg.add_edge(i, j)
            nxg[i][j]['color'] = "black"
        
# for e in d["edges"]:
#     vertexes = d["vertices"]
#     edge_type = e[2]
#     edge_color = "red" if edge_type == EdgeType.HADAMARD else "black"
#     if e[0] not in bounds and e[1] not in bounds:
#         nxg.add_edge(e[0], e[1])
#         nxg[e[0]][e[1]]['color'] = edge_color

pos = nx.spring_layout(nxg, seed=42)  # nice spacing

node_colors = [nxg.nodes[n]['color'] for n in nxg.nodes()]
edge_colors = [nxg[u][v]['color'] for u,v in nxg.edges()]

# Draw the graph
plt.figure(figsize=(10,8))
nx.draw_networkx_nodes(nxg, pos, node_color=node_colors, node_size=700)
nx.draw_networkx_edges(nxg, pos, edge_color=edge_colors, width=2)
# nx.draw_networkx_labels(nxg, pos, font_size=10, font_color='white')

# Add title to the plot
plt.title("5 qubit code", fontsize=16, pad=20)

plt.axis('off')
plt.show()

net = Network(notebook=True, directed=False)
net.from_nx(nxg)


In [ ]:
from collections import defaultdict

class Pauli:

    # (a,b,c) represents i^a * X^b * Z^c
    def __init__ (self, a, b, c):
        self.a = a % 4
        self.b = b % 2
        self.c = c % 2

    def __mul__(self, other):
        s = (self.b * other.c - self.c * other.b) % 2 # commutation factor
        a = (self.a + other.a + 2*s) % 4 # phase
        b = (self.b + other.b) % 2 # X part
        c = (self.c + other.c) % 2 # Z part
        return GraphState.Pauli(a, b, c)

    def __repr__(self):
        phase = [1, 1j, -1, -1j][self.a]
        label = { (0,0):"I", (1,0):"X", (0,1):"Z", (1,1):"Y" }[(self.b,self.c)]
        return f"{phase}*{label}"


inputs = d["inputs"]
adj = d["adjacency_list"]

print(adj[2])

# sets = [set(adj[i]) for i in range(len(adj))]

# element_to_source = [[] for _ in range(len(adj))]

# for i in range(len(adj)):
#     for element in adj[i]:
#         element_to_source[element].append(i)

# print(element_to_source)
# pivots = [-1 for _ in range(len(adj))]

# for i in range(len(element_to_source)):
#     if len(element_to_source[i]) == 1:
#         pivots[i] = element_to_source[i][0]

# # for 

# print(element_to_source)

print(inputs)

pivots = {i : -1 for i in inputs}

out_to_in = {i : -1 for i in range(len(adj)) if i not in inputs}

print(inputs)
print(adj)

for i in range(len(adj)):
    if i not in inputs:
        # print()
        neigh = adj[i]
        count = 0
        input = -1
        for n in neigh:
            if n in inputs:
                count += 1
                input = n
                # print(n)
                out_to_in[i] = n
        if count == 1:
            if pivots[input] == -1:
                pivots[input] = i 

# print(out_to_in)


print("pivots", pivots.values())
print("out to in", out_to_in)



stabilizers = [[Pauli(0,0,0) for _ in range(len(adj) - len(inputs)) ] for _ in range(len(adj) - 2 * len(inputs))]

k = 0
for i in range(len(inputs), len(adj)):
    if i not in pivots.values():
        print("Scanning vertex", i)
        # print(k, i)
        stabilizers[k][i - len(inputs)] *= Pauli(0,1,0)
        print(stabilizers[k])
        for x in adj[i]:
            if x not in inputs:
                stabilizers[k][x - len(inputs)] *= Pauli(0,0,1)
        print(stabilizers[k])
        y = pivots[out_to_in[i]] 
        stabilizers[k][y - len(inputs)] *= Pauli(0,1,0)
        print(stabilizers[k])
        for x in adj[y]:
            if x not in inputs:
                stabilizers[k][x- len(inputs)] *= Pauli(0,0,1)
        print(stabilizers[k])
        k += 1

print(stabilizers)



[1, 3, 0]
[0]
[0]
[[2, 4, 5, 1, 3], [2, 0, 5], [1, 3, 0], [2, 4, 0], [5, 3, 0], [4, 0, 1]]
pivots dict_values([1])
out to in {1: 0, 2: 0, 3: 0, 4: 0, 5: 0}
Scanning vertex 2
[1*I, 1*X, 1*I, 1*I, 1*I]
[1*Z, 1*X, 1*Z, 1*I, 1*I]
[-1*Y, 1*X, 1*Z, 1*I, 1*I]
[-1*Y, -1*Y, 1*Z, 1*I, 1*Z]
Scanning vertex 3
[1*I, 1*I, 1*X, 1*I, 1*I]
[1*I, 1*Z, 1*X, 1*Z, 1*I]
[1*X, 1*Z, 1*X, 1*Z, 1*I]
[1*X, 1*I, 1*X, 1*Z, 1*Z]
Scanning vertex 4
[1*I, 1*I, 1*I, 1*X, 1*I]
[1*I, 1*I, 1*Z, 1*X, 1*Z]
[1*X, 1*I, 1*Z, 1*X, 1*Z]
[1*X, 1*Z, 1*Z, 1*X, 1*I]
Scanning vertex 5
[1*I, 1*I, 1*I, 1*I, 1*X]
[1*Z, 1*I, 1*I, 1*Z, 1*X]
[-1*Y, 1*I, 1*I, 1*Z, 1*X]
[-1*Y, 1*Z, 1*I, 1*Z, -1*Y]
[[-1*Y, -1*Y, 1*Z, 1*I, 1*Z], [1*X, 1*I, 1*X, 1*Z, 1*Z], [1*X, 1*Z, 1*Z, 1*X, 1*I], [-1*Y, 1*Z, 1*I, 1*Z, -1*Y]]
